# 99 — Browser Agent (Playwright)
## Build a ReAct agent that controls a real browser to research the web
⏱ ~60 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/99-browser-agent-playwright/browser_agent_playwright_workbook.ipynb)

Most web scraping is brittle — it breaks the moment a site changes its HTML structure. A **browser agent** sidesteps this entirely: instead of parsing raw HTML, it gives an LLM the same controls a human has (navigate, click, read, scroll) and lets the model figure out how to extract what it needs.

In this workshop you will build a **ReAct browser agent** using [Playwright](https://playwright.dev/) and [LangGraph](https://langchain-ai.github.io/langgraph/). The agent:
1. Opens a real Chromium browser (headless by default)
2. Navigates to any URL
3. Reads page content, clicks links, and fills forms using LangChain's `PlaywrightBrowserToolkit`
4. Reasons about the content with GPT-4o-mini and produces a structured answer

This is the foundation of web-research agents, RPA (Robotic Process Automation) bots, and autonomous testing systems.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — ReAct loop, browser tools, why Playwright |
| 2 | **Setup** — Install deps, configure API key |
| 3 | **Browser Toolkit** — What tools the agent has |
| 4 | **Building the Workflow** — `create_react_agent` + Playwright |
| 5 | **Running the Agent** — End-to-end execution |
| 6 | **Customising Task + Prompt** — Research vs. RPA patterns |
| 7 | **Tracing the ReAct Loop** — Inspecting tool calls |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- Chromium browser (installed by `playwright install chromium`)

### Key References
> Yao et al. 2023 — [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629) — the core loop this agent implements
>
> Yang et al. 2024 — [SWE-agent: Agent-Computer Interfaces Enable Automated Software Engineering](https://arxiv.org/abs/2405.15793) — shows how structured tool interfaces amplify agent capability
>
> [LangChain Playwright Toolkit docs](https://python.langchain.com/docs/integrations/tools/playwright/)
>
> [LangGraph `create_react_agent` docs](https://langchain-ai.github.io/langgraph/reference/prebuilt/#create_react_agent)

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "langchain",
         "langchain-openai",
         "langchain-community",
         "langgraph",
         "playwright",
         "python-dotenv"],
        check=True
    )
    # Install the Chromium browser binary
    subprocess.run(["playwright", "install", "chromium"], check=True)
    print("Colab install complete.")
else:
    print("Local — skipping pip install.")
    print("Ensure you have run: pip install playwright && playwright install chromium")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
if not (bool(key) and key.startswith('sk-')):
    raise RuntimeError("OPENAI_API_KEY is required for the live browser-agent cells.")
print("API key ready: True")

---
## Part 1 — Concepts: The ReAct Loop and Browser Agents

### What is ReAct?

ReAct (Yao et al. 2023) is a prompting pattern that interleaves **reasoning** and **acting** in a tight loop:

```
Thought: I need to find the main heading on the page.
Action:  navigate_browser(url="https://example.com")
Observation: Page loaded. Title: "Example Domain".
Thought: Now I should read the body text.
Action:  get_elements(selector="p")
Observation: "This domain is for use in illustrative examples..."
Thought: I have enough to write the summary.
Final Answer: Example.com is a placeholder domain maintained by IANA...
```

Each iteration the model: **thinks** about what to do, **calls a tool**, receives an **observation**, and decides whether it's done.

### Why Playwright?

| Approach | How it works | Limitation |
|----------|-------------|------------|
| `requests` + BeautifulSoup | Fetches raw HTML | Fails on JS-rendered pages |
| Selenium | Controls browser, older API | Verbose, slower |
| **Playwright** | Controls browser, modern async API | Needs browser binary |
| Headless Chrome (CDP) | Low-level browser protocol | Very complex |

Playwright is the sweet spot: it handles JavaScript-heavy sites, has excellent Python bindings, and LangChain wraps it into a clean `PlaywrightBrowserToolkit`.

### The ReAct Agent Architecture

```
User Task
    │
    ▼
┌─────────────────────────────────┐
│         LangGraph ReAct Loop     │
│                                  │
│  ┌──────────────────────────┐   │
│  │  GPT-4o-mini (LLM node)  │   │
│  │  - Reads messages        │   │
│  │  - Chooses tool OR done  │   │
│  └────────────┬─────────────┘   │
│               │ tool_call        │
│  ┌────────────▼─────────────┐   │
│  │  ToolNode (Playwright)    │   │
│  │  - navigate_browser       │   │
│  │  - get_elements           │   │
│  │  - click_element          │   │
│  │  - extract_text           │   │
│  └────────────┬─────────────┘   │
│               │ observation      │
│               └──► back to LLM   │
└─────────────────────────────────┘
    │
    ▼
Final Answer
```

LangGraph manages the message state — each tool call and its result are appended to the message list, giving the LLM full context of everything that happened.

---
## Part 2 — Setup: Imports and Verifying Playwright

Before building the agent, let's verify Playwright is installed and Chromium is available. The `create_sync_playwright_browser()` utility from LangChain creates a synchronous browser session suitable for use in a standard Python script or notebook.

In [ ]:
# Verify Playwright and LangChain community tools are importable
try:
    from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
    from langchain_community.tools.playwright.utils import create_sync_playwright_browser
    print("PlayWrightBrowserToolkit: OK")
except ImportError as e:
    print(f"Import failed: {e}")
    print("Run: pip install playwright langchain-community && playwright install chromium")

from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
print("LangGraph create_react_agent: OK")

---
## Part 3 — The Browser Toolkit: What Tools Does the Agent Have?

`PlaywrightBrowserToolkit` wraps Playwright's browser into a set of LangChain tools. Let's instantiate a browser and inspect the tools the agent will have access to.

### Common Playwright tools provided by LangChain:

| Tool | What it does |
|------|-------------|
| `navigate_browser` | Go to a URL |
| `navigate_back` | Browser back button |
| `click_element` | Click a CSS selector |
| `get_elements` | Get elements matching a CSS selector |
| `fill_element` | Type text into an input |
| `get_current_page_url` | Return the current URL |
| `extract_text` | Get all visible text on the page |
| `extract_hyperlinks` | Get all links on the page |
| `scroll_element` | Scroll an element or the page |
| `take_screenshot` | Capture a screenshot |

Each tool is described to the LLM with its name, a docstring, and its input schema — so the model knows exactly how to call it.

In [ ]:
# Create a browser and inspect the available tools
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_sync_playwright_browser

browser = create_sync_playwright_browser()
toolkit = PlayWrightBrowserToolkit.from_browser(sync_browser=browser)
tools = toolkit.get_tools()

print(f"Number of tools available: {len(tools)}")
print()
for t in tools:
    print(f"  {t.name:30s} — {t.description[:70]}")

### Tool schemas: what the LLM sees

When the LLM decides to call a tool, it generates JSON that matches the tool's input schema. Let's inspect the schema for `navigate_browser` — one of the most important tools.

In [ ]:
import json

# Find navigate_browser and show its schema
nav_tool = next((t for t in tools if "navigate" in t.name and "back" not in t.name), None)
if nav_tool:
    print(f"Tool name: {nav_tool.name}")
    print(f"Description: {nav_tool.description}")
    print()
    if hasattr(nav_tool, "args_schema") and nav_tool.args_schema:
        schema = nav_tool.args_schema.schema()
        print("Input schema:")
        print(json.dumps(schema, indent=2))
    else:
        print("(no args_schema)")

---
## Part 4 — Building the Workflow

The workflow from `src/workflow.py` is clean and minimal. Let's understand each piece:

### `create_sync_playwright_browser()`
Creates a synchronous Playwright browser session. Synchronous (vs. async) is simpler for notebooks and scripts — the agent blocks while waiting for each page load.

### `PlaywrightBrowserToolkit.from_browser()`
Wraps the browser in a toolkit that exposes it as LangChain-compatible `BaseTool` objects.

### `create_react_agent(llm, tools, state_modifier=...)`
This is LangGraph's prebuilt ReAct agent. It creates a graph with two nodes:
- **LLM node**: calls the model, gets back either a tool call or a final answer
- **Tool node**: executes whatever tool the LLM requested

The `state_modifier` (our system prompt) is injected before every LLM call — it tells the model its role and strategy.

```python
# From src/workflow.py
def create_workflow():
    browser = create_sync_playwright_browser()
    toolkit = PlayWrightBrowserToolkit.from_browser(sync_browser=browser)
    tools = toolkit.get_tools()
    llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
    return create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)
```

Setting `temperature=0` is important for agents — we want deterministic, reliable tool calls, not creative hallucinations.

In [ ]:
# Reproduce the full create_workflow() from src/workflow.py
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_sync_playwright_browser

SYSTEM_PROMPT = (
    "You are a web research assistant. Use the browser tools to navigate "
    "pages, extract their text content, and produce concise summaries. "
    "Always navigate first, then read the page content before summarising."
)

def create_workflow():
    browser = create_sync_playwright_browser()
    toolkit = PlayWrightBrowserToolkit.from_browser(sync_browser=browser)
    tools = toolkit.get_tools()

    llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
    return create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

graph = create_workflow()
print("Workflow created successfully.")
print(f"Graph type: {type(graph).__name__}")

### Visualising the graph

LangGraph can render the compiled graph as a Mermaid diagram. For `create_react_agent` the structure is always the same two-node loop, but it's useful to see it explicitly.

In [ ]:
# Display the ReAct graph structure
try:
    from IPython.display import Image, display
    img = graph.get_graph().draw_mermaid_png()
    display(Image(img))
except Exception as e:
    # Fallback: print Mermaid source
    print(graph.get_graph().draw_mermaid())

---
## Part 5 — Running the Agent

Now the full end-to-end run. From `main.py`, invoking the graph is just three lines:

```python
graph = create_workflow()
result = graph.invoke({"messages": [{"role": "user", "content": TASK}]})
final = result["messages"][-1]
print(final.content)
```

The task string is the only user input. The agent figures out the rest.

### The default task from `src/tools.py`:

```python
TASK = (
    "Navigate to https://example.com, read the full page text, "
    "and write a 2-3 sentence summary of what the page is about."
)
```

We'll run this first, then try a different task.

In [ ]:
# The original task from src/tools.py
TASK = (
    "Navigate to https://example.com, read the full page text, "
    "and write a 2-3 sentence summary of what the page is about."
)

# Run the agent (this opens a real browser and makes real web requests)
result = graph.invoke({"messages": [{"role": "user", "content": TASK}]})

# The final message is the agent's answer
final = result["messages"][-1]
print("=" * 60)
print("AGENT FINAL ANSWER:")
print("=" * 60)
print(final.content)

---
## Part 6 — Customising Task and System Prompt

The system prompt and task string are the two main control surfaces for the agent. Let's explore how to adapt them for different use cases.

### Research mode vs. RPA mode

| Mode | System prompt focus | Task style |
|------|--------------------|-----------|
| **Research** | Summarise, compare, synthesise | Open-ended questions |
| **RPA** | Extract structured data, fill forms | Precise step-by-step instructions |
| **Monitoring** | Detect changes, alert on conditions | Check X and report Y |

The model (gpt-4o-mini at temperature=0) stays the same — only the prompts change.

In [ ]:
# Try a different task — extracting hyperlinks from a page
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_sync_playwright_browser

RESEARCH_SYSTEM_PROMPT = (
    "You are a web research assistant. Navigate pages and extract specific information. "
    "Be precise: report exact text you find on the page, not interpretations. "
    "Always navigate to the page first."
)

HYPERLINK_TASK = (
    "Navigate to https://example.com and list all hyperlinks on the page. "
    "For each link, show the display text and the URL it points to."
)

# Create a fresh browser and workflow for this task
browser2 = create_sync_playwright_browser()
toolkit2 = PlayWrightBrowserToolkit.from_browser(sync_browser=browser2)
tools2 = toolkit2.get_tools()
llm2 = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
graph2 = create_react_agent(llm2, tools2, prompt=RESEARCH_SYSTEM_PROMPT)

result2 = graph2.invoke({"messages": [{"role": "user", "content": HYPERLINK_TASK}]})
final2 = result2["messages"][-1]
print("HYPERLINK EXTRACTION RESULT:")
print(final2.content)

---
## Part 7 — Tracing the ReAct Loop

The real power of inspecting an agent comes from reading its **full message trace** — every thought, every tool call, every observation. LangGraph stores all of this in `result["messages"]`.

### Message types in the trace:

| Message type | When it appears |
|-------------|----------------|
| `HumanMessage` | The initial user task |
| `AIMessage` with `tool_calls` | LLM deciding to use a tool |
| `ToolMessage` | The tool's response (observation) |
| `AIMessage` with `content` (no tool calls) | Final answer |

Counting these messages tells you how many ReAct iterations the agent took.

In [ ]:
# Inspect the full message trace from the first run
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

messages = result["messages"]
print(f"Total messages in trace: {len(messages)}")
print()

for i, msg in enumerate(messages):
    msg_type = type(msg).__name__
    
    if isinstance(msg, HumanMessage):
        print(f"[{i}] HumanMessage: {str(msg.content)[:80]}...")
    elif isinstance(msg, AIMessage) and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{i}] AIMessage → tool_call: {tc['name']}({tc['args']})")
    elif isinstance(msg, ToolMessage):
        content_preview = str(msg.content)[:100].replace("\n", " ")
        print(f"[{i}] ToolMessage (observation): {content_preview}...")
    elif isinstance(msg, AIMessage):
        print(f"[{i}] AIMessage (final answer): {str(msg.content)[:80]}...")
    else:
        print(f"[{i}] {msg_type}: {str(msg)[:60]}")

In [ ]:
# Count ReAct iterations (each tool call + observation = 1 iteration)
tool_calls = sum(
    1 for msg in messages
    if isinstance(msg, AIMessage) and msg.tool_calls
)
observations = sum(1 for msg in messages if isinstance(msg, ToolMessage))

print(f"ReAct iterations: {tool_calls}")
print(f"Tool observations received: {observations}")
print()
print("Tools used in this run:")
for msg in messages:
    if isinstance(msg, AIMessage) and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  - {tc['name']}")

---
## Part 8 — Using Individual Tools Directly

Sometimes you want to test a single browser tool without running the full agent. You can call LangChain tools directly using `.run()` or `.invoke()`. This is useful for:
- Debugging what a tool returns
- Building deterministic pipelines (not agent-driven)
- Testing before wiring into the agent

In [ ]:
# Use individual tools directly (without the agent)
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_sync_playwright_browser

# Create a fresh browser for direct tool use
direct_browser = create_sync_playwright_browser()
direct_toolkit = PlayWrightBrowserToolkit.from_browser(sync_browser=direct_browser)
direct_tools = {t.name: t for t in direct_toolkit.get_tools()}

print(f"Available tools: {list(direct_tools.keys())}")

In [ ]:
# Step 1: Navigate to a page
nav_result = direct_tools["navigate_browser"].invoke({"url": "https://example.com"})
print("Navigate result:", nav_result)

In [ ]:
# Step 2: Extract all visible text
text_result = direct_tools["extract_text"].invoke({})
print("Page text:")
print(text_result)

In [ ]:
# Step 3: Extract all hyperlinks
links_result = direct_tools["extract_hyperlinks"].invoke({"absolute_urls": True})
print("Page hyperlinks:")
print(links_result)

---
## Part 9 — Production Considerations

When moving from a workshop example to a production browser agent, several issues arise:

### 1. Browser lifecycle management

The current code creates a browser per workflow call. In production, you'd pool browsers or use a shared context:

```python
# Pattern: close browser after use
browser = create_sync_playwright_browser()
try:
    toolkit = PlayWrightBrowserToolkit.from_browser(sync_browser=browser)
    graph = create_react_agent(llm, toolkit.get_tools(), prompt=prompt)
    result = graph.invoke({"messages": [{"role": "user", "content": task}]})
finally:
    browser.close()
```

### 2. Async for concurrency

For processing multiple URLs in parallel, switch to async:

```python
from langchain_community.tools.playwright.utils import create_async_playwright_browser
# Use AsyncPlaywrightBrowserToolkit and async graph.ainvoke()
```

### 3. Anti-bot detection

Some sites block headless browsers. Mitigations:
- Use `headless=False` during development to see what's happening
- Set a realistic user agent
- Add delays between actions
- Use residential proxies for scale

### 4. Token cost

Each page's text can be thousands of tokens. `extract_text` returns everything — consider:
- Using `get_elements` with a CSS selector to grab only what you need
- Chunking long pages before sending to the LLM
- Using a cheaper model (gpt-4o-mini is already a good choice)

In [ ]:
# Demonstrate: targeted extraction with get_elements vs full extract_text
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_sync_playwright_browser
import json

compare_browser = create_sync_playwright_browser()
compare_toolkit = PlayWrightBrowserToolkit.from_browser(sync_browser=compare_browser)
compare_tools = {t.name: t for t in compare_toolkit.get_tools()}

# Navigate first
compare_tools["navigate_browser"].invoke({"url": "https://example.com"})

# Full text extraction
full_text = compare_tools["extract_text"].invoke({})

# Targeted: just the <p> elements
try:
    targeted = compare_tools["get_elements"].invoke(
        {"selector": "p", "attributes": ["innerText"]}
    )
except Exception as e:
    targeted = f"(get_elements error: {e})"

print(f"Full text length: {len(full_text)} chars")
print(f"Targeted <p> extraction: {targeted[:200]}")

---
## Part 10 — Complete Example: The Full `main.py` Flow

Let's put it all together exactly as the original `main.py` does it — clean, minimal, end-to-end.

In [ ]:
# Exact reproduction of main.py
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_sync_playwright_browser

load_dotenv()

# --- From src/tools.py ---
TASK = (
    "Navigate to https://example.com, read the full page text, "
    "and write a 2-3 sentence summary of what the page is about."
)

SYSTEM_PROMPT = (
    "You are a web research assistant. Use the browser tools to navigate "
    "pages, extract their text content, and produce concise summaries. "
    "Always navigate first, then read the page content before summarising."
)

# --- From src/workflow.py ---
def create_workflow():
    browser = create_sync_playwright_browser()
    toolkit = PlayWrightBrowserToolkit.from_browser(sync_browser=browser)
    tools = toolkit.get_tools()
    llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
    return create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

# --- From main.py ---
def main():
    graph = create_workflow()
    result = graph.invoke({"messages": [{"role": "user", "content": TASK}]})
    final = result["messages"][-1]
    print(final.content)

main()

---
## Exercises

### Exercise 1 — Multi-page research agent
Modify the task to visit **two different pages** and compare them. For example:
> "Navigate to https://example.com, extract the text. Then navigate to https://iana.org, extract the text. Write a paragraph comparing the purpose of the two sites."

Run the agent with this task and observe how many ReAct iterations it takes compared to the single-page task.

---

### Exercise 2 — Custom system prompt for structured output
Write a new system prompt that instructs the agent to always return its answer as JSON with keys:
```json
{ "url": "...", "title": "...", "summary": "...", "links_count": N }
```
Create a new workflow with this prompt and run it on `https://example.com`.

---

### Exercise 3 — Count the cost
The `result["messages"]` list contains every message. Write a function `estimate_tokens(messages)` that estimates the total token usage by counting characters divided by 4 (rough approximation). Print the estimate for each run you have done in this notebook.

In [ ]:
# ===== ANSWER KEY =====

# --- Exercise 1: Multi-page research agent ---
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_sync_playwright_browser

MULTI_PAGE_TASK = (
    "Navigate to https://example.com, extract the text. "
    "Then navigate to https://www.iana.org, extract the text. "
    "Write a paragraph comparing the purpose of the two sites."
)

SYSTEM_PROMPT = (
    "You are a web research assistant. Use the browser tools to navigate "
    "pages, extract their text content, and produce concise summaries. "
    "Always navigate first, then read the page content before summarising."
)

ex1_browser = create_sync_playwright_browser()
ex1_toolkit = PlayWrightBrowserToolkit.from_browser(sync_browser=ex1_browser)
ex1_tools = ex1_toolkit.get_tools()
ex1_llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
ex1_graph = create_react_agent(ex1_llm, ex1_tools, prompt=SYSTEM_PROMPT)

ex1_result = ex1_graph.invoke({"messages": [{"role": "user", "content": MULTI_PAGE_TASK}]})
print("Exercise 1 — Multi-page comparison:")
print(ex1_result["messages"][-1].content)
print()

from langchain_core.messages import AIMessage, ToolMessage
ex1_iters = sum(1 for m in ex1_result["messages"] if isinstance(m, AIMessage) and m.tool_calls)
print(f"ReAct iterations used: {ex1_iters} (vs ~1 for single-page task)")

In [ ]:
# --- Exercise 2: Structured JSON output ---
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_community.agent_toolkits import PlaywrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_sync_playwright_browser

JSON_SYSTEM_PROMPT = (
    "You are a web data extraction assistant. Use browser tools to visit pages "
    "and extract information. ALWAYS return your final answer as valid JSON with "
    'exactly these keys: {"url": string, "title": string, "summary": string, "links_count": integer}. '
    "Do not include any text outside the JSON object."
)

JSON_TASK = (
    "Navigate to https://example.com. Extract the page title, "
    "summarise the content in one sentence, and count the number of hyperlinks."
)

ex2_browser = create_sync_playwright_browser()
ex2_toolkit = PlayWrightBrowserToolkit.from_browser(sync_browser=ex2_browser)
ex2_tools = ex2_toolkit.get_tools()
ex2_llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
ex2_graph = create_react_agent(ex2_llm, ex2_tools, prompt=JSON_SYSTEM_PROMPT)

ex2_result = ex2_graph.invoke({"messages": [{"role": "user", "content": JSON_TASK}]})
raw_answer = ex2_result["messages"][-1].content
print("Exercise 2 — Structured JSON output:")
print(raw_answer)

# Try to parse it
import json
try:
    parsed = json.loads(raw_answer)
    print()
    print("Parsed successfully:")
    for k, v in parsed.items():
        print(f"  {k}: {v}")
except json.JSONDecodeError:
    print("(Note: model returned prose; adjust prompt to enforce JSON more strictly)")

In [ ]:
# --- Exercise 3: Token cost estimation ---
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def estimate_tokens(messages):
    """Rough token estimate: characters / 4 (industry rule of thumb for English text)."""
    total_chars = 0
    breakdown = {}
    for msg in messages:
        msg_type = type(msg).__name__
        content = str(msg.content) if msg.content else ""
        # Also count tool call content if present
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                content += str(tc)
        chars = len(content)
        total_chars += chars
        breakdown[msg_type] = breakdown.get(msg_type, 0) + chars
    
    token_estimate = total_chars // 4
    return token_estimate, breakdown

# Estimate for the original run
if 'result' in dir():
    tokens, breakdown = estimate_tokens(result["messages"])
    print("Original single-page run:")
    print(f"  Estimated tokens: {tokens}")
    for msg_type, chars in breakdown.items():
        print(f"  {msg_type}: ~{chars // 4} tokens ({chars} chars)")

# Estimate for multi-page run (Exercise 1)
if 'ex1_result' in dir():
    print()
    tokens2, breakdown2 = estimate_tokens(ex1_result["messages"])
    print("Multi-page run (Exercise 1):")
    print(f"  Estimated tokens: {tokens2}")
    for msg_type, chars in breakdown2.items():
        print(f"  {msg_type}: ~{chars // 4} tokens ({chars} chars)")

print()
print("Note: gpt-4o-mini costs $0.15/1M input tokens, $0.60/1M output tokens (as of mid-2025).")
print("A single browser agent run is typically well under 10k tokens.")

---
## Workshop Complete

You have built, run, and inspected a **ReAct browser agent** using Playwright and LangGraph. Here is what you covered:

- **ReAct loop**: the Thought → Action → Observation cycle that drives agentic behaviour
- **PlaywrightBrowserToolkit**: LangChain's wrapper giving agents human-level browser control
- **`create_react_agent`**: LangGraph's prebuilt two-node graph (LLM + ToolNode)
- **Message tracing**: reading `result["messages"]` to see every tool call and observation
- **Direct tool use**: calling browser tools without the agent for debugging and pipelines
- **Production patterns**: async concurrency, browser lifecycle, token cost awareness

### Next Steps

**Next: example 100 — Multi-Agent Supervisor** — coordinate multiple specialised agents under a supervisor that routes tasks to the right worker.

### Further Reading
- Yao et al. 2023 — [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)
- [LangChain Playwright integration docs](https://python.langchain.com/docs/integrations/tools/playwright/)
- [LangGraph prebuilt agents reference](https://langchain-ai.github.io/langgraph/reference/prebuilt/)
- [Playwright Python docs](https://playwright.dev/python/docs/intro)